In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import re

## 1. Define MRT Nodes

In [2]:
# Define nodes
station_codes = pd.read_csv('mrt_lrt_stations.csv')
# exclude all the stations that LINE_COLOR is not grey
nodes = station_codes[station_codes['LINE_COLOR'] != 'Grey']['ALPHANUMERIC_CODE'].unique().tolist()
nodes.insert(nodes.index('EW33') + 1, 'CG0') # insert CG0 because tanah merah has no CG code but it is an interchange to go to expo/CG
nodes.insert(nodes.index('CC29') + 1, 'CE0') # insert CE0 because promenade has no CE code but it is an interchange to go to sentosa/CE
# nodes

### Filter for Downtown Line Stations

In [3]:
# nodes is a list.
# print(type(nodes))
dtl_stations = [station for station in nodes if station.startswith('DT')]
dtl_stations

['DT1',
 'DT2',
 'DT3',
 'DT4',
 'DT5',
 'DT6',
 'DT7',
 'DT8',
 'DT9',
 'DT10',
 'DT11',
 'DT12',
 'DT13',
 'DT14',
 'DT15',
 'DT16',
 'DT17',
 'DT18',
 'DT19',
 'DT20',
 'DT21',
 'DT22',
 'DT23',
 'DT24',
 'DT25',
 'DT26',
 'DT27',
 'DT28',
 'DT29',
 'DT30',
 'DT31',
 'DT32',
 'DT33',
 'DT34',
 'DT35']

### Classify DTL Stations by Demand Tier

In [4]:
# 1. Filter for Weekday, 7am-8am Window and only DT stations
df = pd.read_csv("transport_node_train_202603.csv")
dtl_data = df[
    (df['DAY_TYPE'] == 'WEEKDAY') & 
    (df['TIME_PER_HOUR'] == 7) & 
    (df['PT_CODE'].str.contains('DT'))
].copy()

# 2. Standardize PT_CODE to just the 'DTxx' part
# This turns "NS21 / DT11" into "DT11"
dtl_data['PT_CODE'] = dtl_data['PT_CODE'].str.extract(r'(DT\d+)')

# 3. Calculate Throughput
dtl_data['volume'] = dtl_data['TOTAL_TAP_IN_VOLUME'] + dtl_data['TOTAL_TAP_OUT_VOLUME']

# 4. Sort Numerically (Extract number from 'DT1', 'DT2', etc.)
dtl_data['stn_num'] = dtl_data['PT_CODE'].str.extract(r'(\d+)').astype(int)
dtl_data = dtl_data.sort_values('stn_num')

# 5. Create the sorted volume dictionary
sorted_volumes = dict(zip(dtl_data['PT_CODE'], dtl_data['volume']))

# 6. Divide into Tiers based on volume (High, Mid, Low)
high_thresh = dtl_data['volume'].quantile(0.8)  # Top 20%
# threshold can be changed later. i used a 20% cut-off to tighten tier 1 
# because if tier 1 has too many stations it would be force to stop at too many stations :(
low_thresh = dtl_data['volume'].quantile(0.3)   # Bottom 30%

tier_high = set(dtl_data[dtl_data['volume'] >= high_thresh]['PT_CODE'])
tier_mid = set(dtl_data[(dtl_data['volume'] < high_thresh) & (dtl_data['volume'] >= low_thresh)]['PT_CODE'])
tier_low = set(dtl_data[dtl_data['volume'] < low_thresh]['PT_CODE'])

# Step 7: Manual addition
# These must be Tier 1 to preserve network connectivity
interchanges_dtl = {
    'DT1', 'DT9', 'DT10', 'DT11', 'DT12', 'DT14', 
    'DT15', 'DT16', 'DT19', 'DT26', 'DT32', 'DT35'
}
# Force them into Tier 1
tier_high.update(interchanges_dtl)

# Remove them from other tiers to ensure no duplicates
tier_mid.difference_update(interchanges_dtl)
tier_low.difference_update(interchanges_dtl)

print("Station Volumes:", sorted_volumes)
print("Tier 1:", tier_high)
print("Tier 2:", tier_mid)
print("Tier 3:", tier_low)


Station Volumes: {'DT1': 216813, 'DT2': 25996, 'DT3': 32897, 'DT4': 11434, 'DT5': 58113, 'DT6': 32699, 'DT7': 23770, 'DT8': 43803, 'DT9': 21539, 'DT10': 23752, 'DT11': 167655, 'DT12': 67344, 'DT13': 19718, 'DT14': 75772, 'DT15': 25717, 'DT16': 41395, 'DT17': 44111, 'DT18': 32717, 'DT19': 57668, 'DT20': 19712, 'DT21': 17161, 'DT22': 28894, 'DT23': 16493, 'DT24': 24057, 'DT25': 20786, 'DT26': 63868, 'DT27': 46914, 'DT28': 64299, 'DT29': 39386, 'DT30': 45449, 'DT31': 44461, 'DT32': 195947, 'DT33': 106807, 'DT34': 24156, 'DT35': 38070}
Tier 1: {'DT33', 'DT1', 'DT19', 'DT16', 'DT26', 'DT9', 'DT32', 'DT15', 'DT14', 'DT28', 'DT35', 'DT10', 'DT11', 'DT12'}
Tier 2: {'DT3', 'DT2', 'DT22', 'DT8', 'DT30', 'DT27', 'DT29', 'DT31', 'DT17', 'DT18', 'DT5', 'DT6'}
Tier 3: {'DT21', 'DT4', 'DT7', 'DT13', 'DT23', 'DT34', 'DT20', 'DT24', 'DT25'}


<>:17: SyntaxWarning: invalid escape sequence '\d'
<>:17: SyntaxWarning: invalid escape sequence '\d'
/var/folders/vj/81995_hj5kg3qp6vxrv2yk_w0000gn/T/ipykernel_58568/3606903531.py:17: SyntaxWarning: invalid escape sequence '\d'
  dtl_data['stn_num'] = dtl_data['PT_CODE'].str.extract('(\d+)').astype(int)


## 2. Define Network Arcs

In [5]:
# each node is connected to the next node in the same line, and also to the interchanges
# the nodes have values e.g. NS1, NS2, NS3. NS1 has an arc to NS2, NS2 has an arc to NS3 but NS1 has no arc to NS3.
arcs = []
for i in range(len(nodes) - 1):
    if nodes[i][:2] == nodes[i + 1][:2]:
        arcs.append((nodes[i], nodes[i + 1]))
# print(arcs)
interchanges = [('NS1', 'EW24'), ('NS9', 'TE2'), ('CE0', 'CC4'), ('CE0', 'DT16'),
                ('NS17', 'CC15'), ('NS21', 'DT11'), ('NS22', 'TE14'), 
               ('NS24', 'NE6'), ('NS24', 'CC1'), ('NS25', 'EW13'), 
               ('NS26', 'EW14'), ('NS27', 'TE20'), ('NS27', 'CE2'),
               ('CC22', 'EW21'), ('EW16', 'NE3'), ('EW16', 'TE17'), 
               ('EW12', 'DT14'), ('EW8', 'CC9'), ('EW4', 'CG0'), 
               ('EW2', 'DT32'), ('CG1', 'DT35'), ('CC19', 'DT9'),
               ('DT10', 'TE11'), ('NE7', 'DT12'), ('DT15', 'CC4'),
               ('DT16', 'CE1'), ('DT19', 'NE4'), ('DT26', 'CC10'),
               ('CC29', 'NE1'), ('CC17', 'TE9'), ('CC13', 'NE12'),
               ('CC1', 'NE6'), ('CE2', 'TE20'), ('NE3', 'TE17'), 
               ]
for interchange in interchanges:
    arcs.append(interchange)
# since all arcs are bidirectional, need to include the other direction as well.
for arc in arcs.copy():
    arcs.append((arc[1], arc[0]))

print(arcs)

[('NS1', 'NS2'), ('NS2', 'NS3'), ('NS3', 'NS4'), ('NS4', 'NS5'), ('NS5', 'NS7'), ('NS7', 'NS8'), ('NS8', 'NS9'), ('NS9', 'NS10'), ('NS10', 'NS11'), ('NS11', 'NS12'), ('NS12', 'NS13'), ('NS13', 'NS14'), ('NS14', 'NS15'), ('NS15', 'NS16'), ('NS16', 'NS17'), ('NS17', 'NS18'), ('NS18', 'NS19'), ('NS19', 'NS20'), ('NS20', 'NS21'), ('NS21', 'NS22'), ('NS22', 'NS23'), ('NS23', 'NS24'), ('NS24', 'NS25'), ('NS25', 'NS26'), ('NS26', 'NS27'), ('NS27', 'NS28'), ('EW1', 'EW2'), ('EW2', 'EW3'), ('EW3', 'EW4'), ('EW4', 'EW5'), ('EW5', 'EW6'), ('EW6', 'EW7'), ('EW7', 'EW8'), ('EW8', 'EW9'), ('EW9', 'EW10'), ('EW10', 'EW11'), ('EW11', 'EW12'), ('EW12', 'EW13'), ('EW13', 'EW14'), ('EW14', 'EW15'), ('EW15', 'EW16'), ('EW16', 'EW17'), ('EW17', 'EW18'), ('EW18', 'EW19'), ('EW19', 'EW20'), ('EW20', 'EW21'), ('EW21', 'EW22'), ('EW22', 'EW23'), ('EW23', 'EW24'), ('EW24', 'EW25'), ('EW25', 'EW26'), ('EW26', 'EW27'), ('EW27', 'EW28'), ('EW28', 'EW29'), ('EW29', 'EW30'), ('EW30', 'EW31'), ('EW31', 'EW32'), ('EW3

#### Keep Only Downtown Line Arcs

In [6]:
dtl_arcs = [arc for arc in arcs if arc[0] in dtl_stations and arc[1] in dtl_stations]
dtl_arcs

[('DT1', 'DT2'),
 ('DT2', 'DT3'),
 ('DT3', 'DT4'),
 ('DT4', 'DT5'),
 ('DT5', 'DT6'),
 ('DT6', 'DT7'),
 ('DT7', 'DT8'),
 ('DT8', 'DT9'),
 ('DT9', 'DT10'),
 ('DT10', 'DT11'),
 ('DT11', 'DT12'),
 ('DT12', 'DT13'),
 ('DT13', 'DT14'),
 ('DT14', 'DT15'),
 ('DT15', 'DT16'),
 ('DT16', 'DT17'),
 ('DT17', 'DT18'),
 ('DT18', 'DT19'),
 ('DT19', 'DT20'),
 ('DT20', 'DT21'),
 ('DT21', 'DT22'),
 ('DT22', 'DT23'),
 ('DT23', 'DT24'),
 ('DT24', 'DT25'),
 ('DT25', 'DT26'),
 ('DT26', 'DT27'),
 ('DT27', 'DT28'),
 ('DT28', 'DT29'),
 ('DT29', 'DT30'),
 ('DT30', 'DT31'),
 ('DT31', 'DT32'),
 ('DT32', 'DT33'),
 ('DT33', 'DT34'),
 ('DT34', 'DT35'),
 ('DT2', 'DT1'),
 ('DT3', 'DT2'),
 ('DT4', 'DT3'),
 ('DT5', 'DT4'),
 ('DT6', 'DT5'),
 ('DT7', 'DT6'),
 ('DT8', 'DT7'),
 ('DT9', 'DT8'),
 ('DT10', 'DT9'),
 ('DT11', 'DT10'),
 ('DT12', 'DT11'),
 ('DT13', 'DT12'),
 ('DT14', 'DT13'),
 ('DT15', 'DT14'),
 ('DT16', 'DT15'),
 ('DT17', 'DT16'),
 ('DT18', 'DT17'),
 ('DT19', 'DT18'),
 ('DT20', 'DT19'),
 ('DT21', 'DT20'),
 ('DT22'

## 3. Define Travel-Time Costs
The cost dictionary stores travel time in seconds for each directed arc. Station-to-station timings are read from `station_costs_no_interchanges.csv`, while transfer timings are read from `station_costs_interchanges.csv`.

Source spreadsheet: https://docs.google.com/spreadsheets/d/13CTjSiDdrjvIJvvCso10TWSP5Bf1kZrR77W638BVFkI/edit?gid=987939193#gid=987939193

In [7]:
station_timings_cost = pd.read_csv('station_costs_no_interchanges.csv')
# Map each station to cost given the codes
costs = {}
# print(station_timings_cost.iloc[2]['station_code'])
for index, row in station_timings_cost.iterrows():
    station_code = row['station_code']
    # convert travel_time to a string
    row['travel_time'] = str(row['travel_time'])
    prev_station_code = station_timings_cost.iloc[index-1]['station_code'] if index >= 0 else None
    # convert travel_time from 0:04:25 to number of seconds, which is 4*60 + 25 = 265 seconds
    cost_in_seconds = row['travel_time'].split(':') if row['travel_time'] != '0:00:00' else [0, 0, 0]
    cost_in_seconds = int(cost_in_seconds[0]) * 3600 + (int(cost_in_seconds[1]) * 60 + int(cost_in_seconds[2]))
    if prev_station_code and station_code[:2] == prev_station_code[:2] and cost_in_seconds != 0: # if the station is on the same line as the previous station, then the cost is the travel time between them
        costs[(prev_station_code, station_code)] = cost_in_seconds
        costs[(station_code, prev_station_code)] = cost_in_seconds # since the arcs are bidirectional, the cost is the same in both directions
# print(costs)
# need to manually add the costs for transfer at interchanges
station_timings_cost_interchanges = pd.read_csv('station_costs_interchanges.csv')
station_timings_cost_interchanges = station_timings_cost_interchanges.dropna()
for index, row in station_timings_cost_interchanges.iterrows():
    station_code_1 = row['station_code_1']
    station_code_2 = row['station_code_2']
    # convert travel_time to a string
    row['travel_time'] = str(row['travel_time'])
    cost_in_seconds = row['travel_time'].split(':') if row['travel_time'] != '0:00:00' else [0, 0, 0]
    cost_in_seconds = int(cost_in_seconds[0]) * 3600 + (int(cost_in_seconds[1]) * 60 + int(cost_in_seconds[2]))
    costs[(station_code_1, station_code_2)] = cost_in_seconds
    costs[(station_code_2, station_code_1)] = cost_in_seconds # since the arcs are bidirectional, the cost is the same in both directions

print(costs)

{('TE1', 'TE2'): 116, ('TE2', 'TE1'): 116, ('TE2', 'TE3'): 152, ('TE3', 'TE2'): 152, ('TE3', 'TE4'): 265, ('TE4', 'TE3'): 265, ('TE4', 'TE5'): 183, ('TE5', 'TE4'): 183, ('TE5', 'TE6'): 139, ('TE6', 'TE5'): 139, ('TE6', 'TE7'): 129, ('TE7', 'TE6'): 129, ('TE7', 'TE8'): 164, ('TE8', 'TE7'): 164, ('TE8', 'TE9'): 200, ('TE9', 'TE8'): 200, ('TE9', 'TE11'): 217, ('TE11', 'TE9'): 217, ('TE11', 'TE12'): 190, ('TE12', 'TE11'): 190, ('TE12', 'TE13'): 122, ('TE13', 'TE12'): 122, ('TE13', 'TE14'): 128, ('TE14', 'TE13'): 128, ('TE14', 'TE15'): 142, ('TE15', 'TE14'): 142, ('TE15', 'TE16'): 90, ('TE16', 'TE15'): 90, ('TE16', 'TE17'): 112, ('TE17', 'TE16'): 112, ('TE17', 'TE18'): 102, ('TE18', 'TE17'): 102, ('TE18', 'TE19'): 119, ('TE19', 'TE18'): 119, ('TE19', 'TE20'): 109, ('TE20', 'TE19'): 109, ('TE20', 'TE22'): 164, ('TE22', 'TE20'): 164, ('TE22', 'TE23'): 198, ('TE23', 'TE22'): 198, ('TE23', 'TE24'): 125, ('TE24', 'TE23'): 125, ('TE24', 'TE25'): 132, ('TE25', 'TE24'): 132, ('TE25', 'TE26'): 113, 

#### Keep Only Downtown Line Costs

In [8]:
# a single entry in the cost dictionary is ('TE1', 'TE2'): 116
# take a subset of those costs whose keys are in dtl_arcs, i.e. both stations are in dtl_stations
dtl_costs = {arc: cost for arc, cost in costs.items() if arc in dtl_arcs}
# dtl_costs
print(len(dtl_costs))

68


Sanity check: the DTL has 35 stations, so there are 34 adjacent station pairs and 68 directed arcs after adding both directions.

## 4. Create the Optimization Model
This cell initializes the Gurobi model, decision variables, and `q_j`, the average number of onboard passengers affected by a stop at station `j`. `q_j` is defined here because Constraint 6 uses it before the objective is built.

In [9]:
print(len(dtl_stations))

35


In [10]:
TRAIN_COUNT = 24 # Total trains in an hour
m = gp.Model('minimize total passenger travel time + waiting penalty + operating costs')
STATION_COUNT = len(dtl_stations)
# create a TRAIN_COUNT by STATION_COUNT matrix of binary decision variables, where x[t][j] = 1 if train t stops at station j, and 0 otherwise
x = [[0 for j in range(STATION_COUNT)] for t in range(TRAIN_COUNT)]
# Create decision variables for each station and each train, where x[t][j] = 1 if train t stops at station j, and 0 otherwise
for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        x[t][j] = m.addVar(vtype=GRB.BINARY, name=f'x_{t}_{j}')

y = [0 for t in range(TRAIN_COUNT)]       
# y[t] is binary, 1 if train t is express, 0 if train t is regular, local train
for t in range(TRAIN_COUNT):
    y[t] = m.addVar(vtype=GRB.BINARY, name=f'y_{t}')
z = [0 for j in range(STATION_COUNT)]
# z[j] is binary, 1 if station j is designated as express station, 0 otherwise
for j in range(STATION_COUNT):
    z[j] = m.addVar(vtype=GRB.BINARY, name=f'z_{j}')

u = [0 for j in range(STATION_COUNT)]
# u[j] >= 0 is the service shortfall at station j
# if u[j] = 0, station j gets enough stopping trains
# if u[j] > 0, station j is under-served and incurs a penalty
for j in range(STATION_COUNT):
    u[j] = m.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f'u_{j}')

print(len(x[0]))

# q[j] = average onboard passengers affected when a train stops at station j.
# It is defined here so Constraint 6 can use q before the objective section.
df_raw = pd.read_csv('origin_destination_train_202603.csv')

def get_final_q_list(df, dtl_stations, num_weekdays=22, num_trains=TRAIN_COUNT):
    mask = (
        (df['DAY_TYPE'] == 'WEEKDAY') &
        (df['TIME_PER_HOUR'] == 7) &
        (df['ORIGIN_PT_CODE'].str.contains('DT', na=False)) &
        (df['DESTINATION_PT_CODE'].str.contains('DT', na=False))
    )
    df_filtered = df[mask].copy()

    def extract_dt(val):
        match = re.search(r'DT\d+', str(val))
        return match.group(0) if match else None

    df_filtered['ORIGIN_CLEAN'] = df_filtered['ORIGIN_PT_CODE'].apply(extract_dt)
    df_filtered['DEST_CLEAN'] = df_filtered['DESTINATION_PT_CODE'].apply(extract_dt)

    q_list = []
    for i, current_stat in enumerate(dtl_stations):
        before_stations = dtl_stations[:i]
        at_or_after_stations = dtl_stations[i:]

        onboard_mask = (
            df_filtered['ORIGIN_CLEAN'].isin(before_stations) &
            df_filtered['DEST_CLEAN'].isin(at_or_after_stations)
        )

        total_load = df_filtered.loc[onboard_mask, 'TOTAL_TRIPS'].sum()
        q_avg = (total_load / num_weekdays) / num_trains
        q_list.append(round(q_avg, 4))

    return q_list

q = get_final_q_list(df_raw, dtl_stations)
print(f'q length: {len(q)}')
print(q)

Set parameter Username
Set parameter LicenseID to value 2768791
Academic license - for non-commercial use only - expires 2027-01-22
35


## 5. Constraints
### Constraint 1: Terminal Stations Must Always Be Served
All trains must stop at the first and last DTL stations, so terminals cannot be skipped by express services.

In [11]:
# station DT1 and DT35 must be served by all trains, express or regular, and cannot be skipped.

for t in range(TRAIN_COUNT):
    m.addConstr(x[t][0] == 1)
    m.addConstr(x[t][STATION_COUNT - 1] == 1)

### Constraint 2: Local Trains Stop at Every Station
If train `t` is local (`y_t = 0`), then it must stop at every station.

In [12]:
# add constraint that says if a train is local, it MUST stop at every station, i.e. xtj >= 1 - yt for all t, for all j

for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        m.addConstr(x[t][j] >= 1 - y[t]) 

### Constraint 3A: Mandatory Interchange Stops
This cell makes DTL interchange stations mandatory stops for all trains. It represents a network-connectivity policy for choosing mandatory stops.

In [13]:
M = set()
for interchange in interchanges:
    if interchange[0] in dtl_stations:
        M.add(interchange[0])
    elif interchange[1] in dtl_stations:
        M.add(interchange[1])

station_to_idx = {f"DT{i}": i - 1 for i in range(1, STATION_COUNT + 1)}
for t in range(TRAIN_COUNT):
    for station in M:
        m.addConstr(x[t][station_to_idx[station]] == 1, name=f"mandatory_stop_{t}_{station}")

### Constraint 3B: Mandatory Tier 1 Stops
This cell makes all Tier 1 stations mandatory stops for all trains. In the current notebook, this is the main mandatory-stop set used later as `M`; if both Constraint 3A and 3B are run, the resulting mandatory-stop constraints are cumulative.

In [28]:
M = list(tier_high) 
station_to_idx = {code: i for i, code in enumerate(dtl_stations)}

for t in range(TRAIN_COUNT):
    for station in M:
        j = station_to_idx[station]
        m.addConstr(
            x[t][j] == 1, 
            name=f"C3_MandatoryStop_Train{t}_{station}"
        )

print(f"Stations in M: {sorted(M)}")

Stations in M: ['DT1', 'DT10', 'DT11', 'DT12', 'DT14', 'DT15', 'DT16', 'DT19', 'DT26', 'DT28', 'DT32', 'DT33', 'DT35', 'DT9']


### Constraint 4: Limit the Number of Express Trains
At most `K` trains in the one-hour period can be designated as express trains.

In [15]:
# Let K be the max number of express trains, where K < TRAIN_COUNT. TODO.
K = 6 # Explanation in report

m.addConstr(gp.quicksum(y[j] for j in range(TRAIN_COUNT)) <= K)

<gurobi.Constr *Awaiting Model Update*>

### Constraint 5: Express Trains Stop Only at Express Stations
If train `t` is express (`y_t = 1`), it can only stop at stations selected into the express stopping pattern (`z_j = 1`). Local trains are not restricted by this constraint.

In [16]:
# xtj ≤ zj + (1 – yt) ∀ t ∈ T, j ∈ S
for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        m.addConstr(x[t][j] <= z[j] + (1 - y[t]))

### Constraint 6: Limit Skipped Stations Within Each Bay
The DTL is split into bays between mandatory stops. For each bay, the model limits how many stations an express train can skip, using `q_j` so high-impact stations are less likely to be skipped.

In [27]:
# ∑(1 - x[t][j]) <= L_b y[t] for each bay b and train t.
# L_b is calculated bay-by-bay using the q[j] values defined earlier.

# Indexes for mandatory stop stations
m_indices = sorted([station_to_idx[s] for s in M])

# Include terminal stations so the full line is split into bays between mandatory stops.
if 0 not in m_indices:
    m_indices.insert(0, 0)
if (STATION_COUNT - 1) not in m_indices:
    m_indices.append(STATION_COUNT - 1)

m_indices = sorted(set(m_indices))

# Section the line into bays. Each bay contains stations between two mandatory stops.
bays_indices = []
for i in range(len(m_indices) - 1):
    start = m_indices[i] + 1
    end = m_indices[i + 1]
    if start < end:
        bays_indices.append(list(range(start, end)))

# Add up stations that can be skipped in order of increasing q[j]
# until the exclusion threshold for that bay is reached.
exlvl = 0.5  # change to 0.2 or 0.4 depending on the assumption
bay_skip_limits = {}

for b_idx, bay in enumerate(bays_indices):
    bay_q_values = sorted(q[j] for j in bay)
    total_bay_q = sum(bay_q_values)
    current_sum = 0
    lb_limit = 0

    for val in bay_q_values:
        if total_bay_q > 0 and current_sum + val <= exlvl * total_bay_q:
            current_sum += val
            lb_limit += 1
        else:
            break

    bay_skip_limits[b_idx] = lb_limit

    for t in range(TRAIN_COUNT):
        m.addConstr(
            gp.quicksum(1 - x[t][j] for j in bay) <= lb_limit * y[t],
            name=f"bay_limit_{b_idx}_train_{t}"
        )

print('Bay skip limits:', bay_skip_limits)

### Constraint 7: Service Coverage Target
`H_j` is the target number of trains that should stop at station `j`. If fewer than `H_j` trains stop there, the shortfall is captured by `u_j` and penalized in the objective.

In [30]:
H_list = []

for stn in dtl_data['PT_CODE']:
    if stn in tier_high:
        H_list.append(TRAIN_COUNT)         # 100% Target (24 trains) # TARGETS can be adjusted later, but tier 1 should have 100%
    elif stn in tier_mid:
        H_list.append(int(TRAIN_COUNT * 0.75)) # 75% Target (18 trains)
    else:
        H_list.append(int(TRAIN_COUNT * 0.50)) # 50% Target (12 trains)

H = H_list

for j in range(STATION_COUNT):
    m.addConstr(gp.quicksum(x[t][j] for t in range(TRAIN_COUNT)) + u[j] >= H[j]) 
    # if the number of trains stopping at station j is less than the ideal number H[j], 
    # then uj will be positive and will pay a penalty in the objective function. if the number of trains stopping at station j is greater than or equal to H[j], then uj will be 0 and there will be no penalty in the objective function.

print("Hj for each station j:", H)


Hj for each station j: [24, 18, 18, 12, 18, 18, 12, 18, 24, 24, 24, 24, 12, 24, 24, 24, 18, 18, 24, 12, 12, 18, 12, 12, 12, 24, 18, 24, 18, 18, 18, 24, 24, 12, 24]


### Constraint 8: Express Operating Budget
`F_t` represents the additional operating cost of making train `t` express. The total express operating cost must stay within the budget cap.

In [19]:
# let F[t] be additional operating cost if train t is express.
# we have a budget cap, e.g. for extra signage, timetable etc. TODO.

F = [100 for _ in range(TRAIN_COUNT)]  # Example values, replace with actual costs
BUDGET_CONSTRAINT = 5000 # Example value, replace with actual budget constraint
m.addConstr(gp.quicksum(F[t] * y[t] for t in range(TRAIN_COUNT)) <= BUDGET_CONSTRAINT)


<gurobi.Constr *Awaiting Model Update*>

### Constraint 9: All Express Trains Follow the Same Stop Pattern
Every express train uses the same selected express stations. For example, if the express pattern is A-C-E, then every express train follows A-C-E.

In [20]:
# All express trains must follow the same skip-stop route

for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        # If train t is express, it must stop if station j is an express station
        m.addConstr(x[t][j] >= z[j] - (1 - y[t]), name=f"express_lower_{t}_{j}")

### Constraint 10: Terminals and Mandatory Stops Must Be Express Stations
Terminal stations and mandatory stations in set `M` must be included in the express stopping pattern.

In [21]:
# Force terminal stations to be in the express set
m.addConstr(z[0] == 1, name="terminal_start_in_express_set")
m.addConstr(z[STATION_COUNT - 1] == 1, name="terminal_end_in_express_set")

# Force mandatory stations to be in the express set
station_to_idx = {f"DT{i}": i - 1 for i in range(1, STATION_COUNT + 1)}

for station in M:
    m.addConstr(z[station_to_idx[station]] == 1, name=f"mandatory_express_{station}")

### Constraint 11 Optional: Limit the Number of Express Stations
This optional constraint prevents the express pattern from including every station. Without it, the model can choose an express pattern that is identical to the local service.

In [ ]:
# OPTIONAL. Uncomment if we want to limit the number of express stations, which will in turn limit the number of possible skip-stop patterns for the express trains. TODO.
# m.addConstr(gp.quicksum(z[j] for j in range(STATION_COUNT)) <= STATION_COUNT - 1, name="limit_express_stations")

## 6. Objective Function and Optimization

The objective minimizes the weighted total cost of the service plan:

`travel_time_cost + underservice_cost + express_cost`

The first term is the passenger travel-time cost of stopping trains. If train `t` stops at station `j`, the model pays `k * q_j`, where `k` is the stop-time penalty from deceleration, dwell time, and acceleration, and `q_j` is the estimated number of onboard passengers affected by that stop.

The second term is the under-service penalty. If station `j` receives fewer stopping trains than its target `H_j`, the shortfall is `u_j`. The model pays `p_j * u_j`, where `p_j` is measured in passenger-minutes per missed stopping train and is higher for busier, more central, and interchange stations.

The third term is the additional operating cost of express trains. `F_t * y_t` captures scheduling, information, and operational complexity when train `t` is run as an express train.

In [22]:
# Penalty of under-service p_j
# Logic:
# 1) use actual weekday 7-8am DTL station usage from transport_node_train_202603.csv
# 2) higher penalty near the CBD
# 3) higher penalty at interchange stations
# 4) one unit of u_j means one fewer stopping train than the target H_j

NUM_WEEKDAYS = 22
CBD_STATIONS = {'DT16', 'DT17', 'DT18', 'DT19'}
INTERCHANGE_STATIONS = {'DT1', 'DT9', 'DT10', 'DT11', 'DT12', 'DT14', 'DT15', 'DT16', 'DT19', 'DT26', 'DT32', 'DT35'}

penalty_df = dtl_data[['PT_CODE', 'TOTAL_TAP_IN_VOLUME', 'TOTAL_TAP_OUT_VOLUME', 'volume']].copy()

# Average number of station users in the study hour on a weekday.
penalty_df['avg_hourly_station_users'] = penalty_df['volume'] / NUM_WEEKDAYS

# Map target service levels H_j; H[j] is aligned with dtl_stations[j].
H_map = dict(zip(dtl_stations, H))
penalty_df['H'] = penalty_df['PT_CODE'].map(H_map)

# CBD proximity multiplier based on line-distance to downtown-core stations.
station_idx = {s: i for i, s in enumerate(dtl_stations)}
penalty_df['dist_to_cbd'] = penalty_df['PT_CODE'].map(
    lambda s: min(abs(station_idx[s] - station_idx[c]) for c in CBD_STATIONS if c in station_idx)
)
max_dist = penalty_df['dist_to_cbd'].max()
penalty_df['cbd_mult'] = 1 + 0.30 * (1 - penalty_df['dist_to_cbd'] / max_dist)

# Interchange bonus.
penalty_df['interchange_mult'] = np.where(
    penalty_df['PT_CODE'].isin(INTERCHANGE_STATIONS), 1.25, 1.0
)

# Approximate users affected by one missed stop and the extra wait they incur.
penalty_df['users_per_missed_stop'] = penalty_df['avg_hourly_station_users'] / penalty_df['H']
penalty_df['extra_wait_per_missed_stop_min'] = 60 / penalty_df['H']

# Final under-service penalty in passenger-minutes per unit of u_j.
penalty_df['p_j'] = (
    penalty_df['users_per_missed_stop']
    * penalty_df['extra_wait_per_missed_stop_min']
    * penalty_df['cbd_mult']
    * penalty_df['interchange_mult']
).round(2)

# Force the same order as the optimization variables: p[j] must match dtl_stations[j].
penalty_df = penalty_df.set_index('PT_CODE').loc[dtl_stations].reset_index()
p = penalty_df['p_j'].tolist()

display(
    penalty_df[['PT_CODE', 'avg_hourly_station_users', 'H', 'cbd_mult', 'interchange_mult', 'p_j']]
)
print(p)

### Check `q_j` Values
This display cell shows the passenger-impact weight `q_j` for each station. The values are already defined earlier so Constraint 6 can use them.

In [25]:
# q was already defined before Constraint 6.
# This cell only displays q[j] in station order for checking/reporting.
q_table = pd.DataFrame({
    'station': dtl_stations,
    'q_j': q,
})

display(q_table)
print(f'STATION_COUNT: {len(q)}')
print(q)

STATION_COUNT: 35
[0.0, 153.5436, 154.3182, 173.8731, 182.1326, 216.8144, 218.1402, 213.3087, 204.4489, 202.6686, 204.1174, 196.7386, 183.9034, 174.2254, 153.6534, 133.6951, 117.0663, 73.2121, 49.392, 49.9186, 49.5928, 52.3466, 70.1439, 72.8314, 83.1989, 88.5398, 96.9962, 91.4129, 88.2576, 97.7765, 101.9735, 102.1193, 100.1648, 67.5265, 52.6951]


In [26]:

k = 1.45
# k = 1.45 means that every stop costs each passengers 1.45  minutes
travel_time_cost = k * gp.quicksum(
    q[j] * x[t][j]
    for t in range(TRAIN_COUNT)
    for j in range(STATION_COUNT)
)

underservice_cost = gp.quicksum(
    p[j] * u[j]
    for j in range(STATION_COUNT)
)

express_cost = gp.quicksum(
    F[t] * y[t]
    for t in range(TRAIN_COUNT)
)

m.setObjective(
    travel_time_cost + underservice_cost + express_cost,
    GRB.MINIMIZE
)

m.optimize()


if m.status == GRB.OPTIMAL:
    print("Optimal objective value:", m.objVal)
else:
    print("No optimal solution found. Status code:", m.status)
# # Print solution (change later, may be wrong)
# print("Optimal objective value:", m.objVal)
# print("Train stop patterns:")
# for t in range(TRAIN_COUNT):
#     stops = [j for j in range(STATION_COUNT) if x[t][j].X > 0.5]
#     print(f"Train {t}: {stops}")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.2.0 25C56)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2874 rows, 934 columns and 7912 nonzeros (Min)
Model fingerprint: 0xedd87344
Model has 875 linear objective coefficients
Variable types: 35 continuous, 899 integer (899 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [1e+01, 3e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+03]

Found heuristic solution: objective 145208.19318
Presolve removed 1337 rows and 385 columns
Presolve time: 0.01s
Presolved: 1537 rows, 549 columns, 4080 nonzeros
Variable types: 0 continuous, 549 integer (549 binary)

Root relaxation: objective 1.343008e+05, 359 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time



## 7. Optional Extension: Flow and Capacity Formulation
The cells below are not part of the current skip-stop optimization model. They can be used later if the project is extended into a full network flow formulation.

### Placeholder for Future Model Extensions

### Arc Capacities
For a future flow model, rail arcs can be assigned train capacity based on the number of trains serving that line. Transfer arcs can be treated as very high capacity because walking transfers are not limited by train vehicle capacity.

Reference article: https://medium.com/@simpletan/deep-dive-of-singapores-rail-network-passenger-loads-e331d3b9626b

In [ ]:
capacities = {}

### Supply and Demand at Each Node
This is a placeholder for a future demand model. One possible approach is to define station-level supply and demand based on distance from the CBD and the time period being studied.

In [ ]:
supply = {
    1: GRB.INFINITY,
    2: 0,
    3: 0,
    4: 0
}
# just an example of supply. change later

### Flow Conservation Constraints
These commented constraints are placeholders for a future network flow model.

In [ ]:
# flow = m.addVars(arcs,ub=[capacities[arc] for arc in arcs])
# selling_amount = m.addVar(name="selling_amount")

# # Flow conservation constraints at nodes 
# for i in supply:
#     if (i == 1):
#         m.addConstr(flow.sum(i, '*') == selling_amount)
#     elif (i == 5):
#         m.addConstr(flow.sum('*', i) == selling_amount)
#     else:
#         m.addConstr(flow.sum(i, '*') - flow.sum('*', i) == 0)
